# Computer Vision 2026-2027 - NOVA FCT

## Project Stage 1 - Shot boundary detection and recurring scenes

**Authors**:
- 73803, Rogério Barros Maia Lopes Soares, rbm.soares@campus.fct.unl.pt
- 74514, Guilherme Simões, gaq.simoes@campus.fct.unl.pt

## 1. Introduction

We detect hard cuts in edited video and group shots of the same scene, using only classical tools from the labs
(histograms, edges, local features). Each signal d(t) compares frames t-1 and t, with d(0) = 0, and a cut is
predicted at t when d(t) > T.

Frames are resized from 640x360 to 320x180: global colour, intensity and edge structure are preserved with four
times fewer pixels, which keeps the whole program inside the 5-minute limit. All design choices (thresholds, bins,
combination rule) are made on the development videos only (dev_01, dev_02); the test videos are scored once, at
the end.

In [ ]:
import json
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

import metrics, plotting, signals, tp1

INPUT, OUTPUT = "../input", "../output"
os.makedirs(OUTPUT, exist_ok=True)

DEV = ["dev_01", "dev_02"]
gts = {n: json.load(open(f"{INPUT}/{n}.json")) for n in DEV}
n_flashes = sum(len(g["distractors"]) for g in gts.values())

BINS = [8, 16, 32, 64]
sigs, hist_by_bins = {}, {}
for n in DEV:
    frames = tp1.read_frames(f"{INPUT}/{n}.mp4")
    assert len(frames) == gts[n]["num_frames"]
    sigs[n] = signals.compute_signals(frames)                    # {"pixel", "histogram", "ecr"}
    hist_by_bins[n] = {b: signals.histogram_signal(frames, b) for b in BINS}
    del frames

for name, d in sigs["dev_01"].items():
    print(f"{name:10s} length = {len(d)}, d[0] = {d[0]}, max = {d.max():.3f}")

## 2. Part 1 - Frame dissimilarity and hard cuts

Three signals, each comparing frames t-1 and t:
- **Pixel difference** (baseline): mean absolute grey-level difference (computed in int16 to avoid uint8 wrap-around).
- **Histogram**: chi-square distance between HSV histograms, each channel normalised.
- **Edge Change Ratio**: Canny edges (100/200), 5x5 dilation, maximum of entering and exiting edge ratios; a frame
  without edges gets ratio 0, so there is no division by zero.

The plots below show each signal over the full videos, with ground-truth cuts (red) and flashes (orange).


In [ ]:
for n in DEV:
    fig = plotting.plot_signals(sigs[n], gts[n])
    for ax in fig.axes:
        ax.legend(loc="upper right", fontsize=7)
    plt.show()

In [ ]:
gt, d = gts["dev_01"], sigs["dev_01"]
shot = gt["shots"][0]
examples = [("cut", gt["transitions"][0]["start"]),
            ("inside a shot", (shot["start"] + shot["end"]) // 2),
            ("flash start", gt["distractors"][0]["start"])]

frames = tp1.read_frames(f"{INPUT}/dev_01.mp4")
imgs = {i: frames[i] for i in {t + k for _, t in examples for k in (-1, 0)}}
del frames

fig, axes = plt.subplots(len(examples), 2, figsize=(8, 7))
for row, (label, t) in zip(axes, examples):
    for ax, k in zip(row, (-1, 0)):
        ax.imshow(cv2.cvtColor(imgs[t + k], cv2.COLOR_BGR2RGB))
        ax.set_title(f"{label}: frame {t + k}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

print(f"{'pair (t-1, t)':24s} {'pixel':>8s} {'histogram':>10s} {'ECR':>6s}")
for label, t in examples:
    print(f"{label + ', t=' + str(t):24s} {d['pixel'][t]:8.2f} {d['histogram'][t]:10.3f} {d['ecr'][t]:6.3f}")

### Choosing thresholds (Task 3)

A cut is declared when d(t) > T. For each signal we try 150 thresholds (quantiles of the signal, because the three
signals have very different ranges), compute precision, recall and F1 with the given `metrics` functions on dev_01
and dev_02 together, and keep the threshold with the best F1. Detections at most 2 frames apart (the metric
tolerance) are merged into one cut, keeping the highest value, since a cut often stays high for two frames and each
extra detection would count as a false positive.

In [ ]:
def cuts_from_mask(mask, score, merge_gap=2):
    """Frames where mask is True, with detections at most merge_gap frames apart merged (same code as tp1.py)."""
    return [int(t) for t in tp1.merge_nearby(np.flatnonzero(mask), score, gap=merge_gap)]


def detect_cuts(d, T, merge_gap=2):
    return cuts_from_mask(d > T, d, merge_gap)


def evaluate_cuts(pred_by_video):
    results = []
    for n in DEV:
        pred = {"transitions": [{"type": "cut", "start": t, "end": t} for t in pred_by_video[n]]}
        results.append(metrics.evaluate_video(gts[n], pred))
    return metrics.sum_counts(results)


def candidate_thresholds(signal_by_video, n=150):
    values = np.concatenate([signal_by_video[v] for v in DEV])
    return np.unique(np.quantile(values, np.linspace(0.90, 0.999, n)))


def sweep(signal_by_video, merge_gap=2):
    rows = []
    for T in candidate_thresholds(signal_by_video):
        counts = evaluate_cuts({v: detect_cuts(signal_by_video[v], T, merge_gap) for v in DEV})
        p, r, f1 = metrics.precision_recall_f1(*counts["cut"])
        rows.append((float(T), p, r, f1, counts["flash_fa"]))
    return rows


def best_row(rows):
    return max(rows, key=lambda row: row[3])

### Histogram: colour space and bins

HSV separates colour (H, S) from brightness (V), so it should react less to lighting changes than RGB. We compare
the number of bins by the best dev F1 of each variant; the table shows that the number of bins has almost no effect,
so the choice among near-equal values is not significant.

In [ ]:
print(f"{'bins':>5s} {'threshold':>9s} {'P':>6s} {'R':>6s} {'F1':>6s}")
bins_best = {}
for b in BINS:
    bins_best[b] = best_row(sweep({v: hist_by_bins[v][b] for v in DEV}))
    T, p, r, f1, fa = bins_best[b]
    print(f"{b:5d} {T:9.4f} {p:6.3f} {r:6.3f} {f1:6.3f}")

best_bins = max(bins_best, key=lambda b: bins_best[b][3])
for v in DEV:
    sigs[v]["histogram"] = hist_by_bins[v][best_bins]
print("\nchosen number of bins:", best_bins)

In [ ]:
SIGNALS = ["pixel", "histogram", "ecr"]
sweeps = {s: sweep({v: sigs[v][s] for v in DEV}) for s in SIGNALS}
best = {s: best_row(rows) for s, rows in sweeps.items()}

print(f"{'signal':10s} {'threshold':>9s} {'precision':>9s} {'recall':>7s} {'F1':>6s} {'flash-FA':>9s}")
for s in SIGNALS:
    T, p, r, f1, fa = best[s]
    print(f"{s:10s} {T:9.4f} {p:9.3f} {r:7.3f} {f1:6.3f} {fa:>5d}/{n_flashes}")

plotting.plot_pr_curve({s: [(T, p, r) for T, p, r, *_ in rows] for s, rows in sweeps.items()},
                       title="dev videos: cut detection", path=f"{OUTPUT}/pr_curve.png")
plt.show()

In [ ]:
def busiest_window(gt, length=750):
    events = [t["start"] for t in gt["transitions"]] + [f["start"] for f in gt["distractors"]]
    starts = range(0, max(1, gt["num_frames"] - length), 25)
    return max(starts, key=lambda s: sum(s <= e < s + length for e in events))


start = busiest_window(gts["dev_01"])
end = min(start + 750, gts["dev_01"]["num_frames"])
fig = plotting.plot_signals({s: sigs["dev_01"][s] for s in SIGNALS}, gts["dev_01"],
                            start=start, end=end, path=f"{OUTPUT}/signals_dev_01.png")
for ax, s in zip(fig.axes, SIGNALS):
    ax.axhline(best[s][0], color="gray", ls=":", label=f"best threshold = {best[s][0]:.3g}")
    ax.legend(loc="upper right", fontsize=7)
plt.show()

### Combining signals

We also combine histogram and ECR: AND (both above their thresholds) or OR (at least one). Both thresholds are
chosen together by a grid search that maximises dev F1. The final Part 1 configuration is the best of the three
signals alone and the two combinations.

In [ ]:
def detect_combined(sig_v, thresholds, mode, merge_gap=2):
    tests = [sig_v[s] > T for s, T in thresholds.items()]
    mask = np.logical_and.reduce(tests) if mode == "and" else np.logical_or.reduce(tests)
    score = sum(sig_v[s] / T for s, T in thresholds.items())
    return cuts_from_mask(mask, score, merge_gap)


grid_h = candidate_thresholds({v: sigs[v]["histogram"] for v in DEV}, n=30)
grid_e = candidate_thresholds({v: sigs[v]["ecr"] for v in DEV}, n=30)

combos = {}
for mode in ("and", "or"):
    best_c = None
    for Th in grid_h:
        for Te in grid_e:
            th = {"histogram": float(Th), "ecr": float(Te)}
            counts = evaluate_cuts({v: detect_combined(sigs[v], th, mode) for v in DEV})
            p, r, f1 = metrics.precision_recall_f1(*counts["cut"])
            if best_c is None or f1 > best_c[3]:
                best_c = (th, p, r, f1, counts["flash_fa"])
    combos[mode] = best_c

print(f"{'configuration':22s} {'thresholds':34s} {'P':>6s} {'R':>6s} {'F1':>6s} {'flash-FA':>9s}")
for s in SIGNALS:
    T, p, r, f1, fa = best[s]
    print(f"{s + ' alone':22s} {s + '=%.4f' % T:34s} {p:6.3f} {r:6.3f} {f1:6.3f} {fa:>5d}/{n_flashes}")
for mode, (th, p, r, f1, fa) in combos.items():
    text = ", ".join(f"{k}={v:.4f}" for k, v in th.items())
    print(f"{mode.upper() + ': histogram + ecr':22s} {text:34s} {p:6.3f} {r:6.3f} {f1:6.3f} {fa:>5d}/{n_flashes}")

In [ ]:
print(f"{'merge_gap':>9s} {'P':>6s} {'R':>6s} {'F1':>6s}   (histogram signal, threshold re-tuned)")
for gap in (0, 2):
    T, p, r, f1, fa = best_row(sweep({v: sigs[v]["histogram"] for v in DEV}, merge_gap=gap))
    print(f"{gap:9d} {p:6.3f} {r:6.3f} {f1:6.3f}")

In [ ]:
candidates = [(f"{s} alone", {s: best[s][0]}, "and", best[s][3]) for s in SIGNALS]
candidates += [(f"{m.upper()}: histogram + ecr", combos[m][0], m, combos[m][3]) for m in combos]
label, thresholds, mode, f1 = max(candidates, key=lambda c: c[3])

CONFIG = {"bins": best_bins, "thresholds": thresholds, "mode": mode, "merge_gap": 2}
print("chosen:", label, f"(dev F1 = {f1:.3f})")
print("CONFIG =", CONFIG)
predicted, counts = {}, {}
for v in DEV:
    predicted[v] = detect_combined(sigs[v], thresholds, mode)
    counts[v] = metrics.evaluate_video(gts[v], {"transitions": [{"type": "cut", "start": t, "end": t}
                                                                 for t in predicted[v]]})
print()
metrics.print_table([(v, counts[v]) for v in DEV] + [("DEV", metrics.sum_counts(list(counts.values())))])
print("(scene columns are 0: scenes are Part 3)")

In [ ]:
tol = metrics.TOLERANCE
for v in DEV:
    gt_cuts = [t["start"] for t in gts[v]["transitions"]]
    pred = predicted[v]
    missed = [g for g in gt_cuts if all(abs(g - p) > tol for p in pred)]
    false_alarms = [p for p in pred if all(abs(g - p) > tol for g in gt_cuts)]
    on_flash = [p for p in false_alarms
                if any(f["start"] - tol <= p <= f["end"] + tol for f in gts[v]["distractors"])]
    print(f"{v}: missed cuts       {missed}")
    print(f"{v}: false alarms      {false_alarms}")
    print(f"{v}:   of which flashes {on_flash}")

### Part 1 analysis

- **Flashes are the main problem.** Every signal spikes on flashes about as strongly as on real cuts, so any
  threshold that finds the cuts also fires on the flashes. Almost all remaining false alarms are flashes, which is
  what Part 2 targets.
- **Robustness to motion.** The histogram is flat inside shots; pixel difference and ECR rise during camera motion
  and fast movement, which lowers their precision.
- **Hard cuts.** Cuts between shots with similar colours give almost no histogram response (see the missed cut in
  the failure analysis), while pixel difference and ECR still react.
- **Merging detections** within 2 frames clearly improves precision; combining signals gives a smaller gain, which
  may not transfer to the test videos because two thresholds are tuned on few cuts.

## 3. Part 2 - False alarms

A flash, fast camera motion, or an object crossing the frame can raise the dissimilarity signal inside a single shot. Part 2 starts from the histogram signal of Part 1 on its own (the best single signal), with its best threshold and the same merging of nearby detections as in `tp1.py`: the frames `t` where the signal exceeds the threshold are only candidates.

Each candidate is checked with SIFT. The two frames compared are `t - 4` and `t + 4`. The gap is 4 because a flash lasts up to 3 frames, so neither frame is the flash itself. Descriptors are matched with a brute-force matcher and kept only when they pass Lowe's ratio test at 0.75. If those two frames still show the same content, the candidate is a false alarm and is rejected: the candidate is rejected when it has at least `min_matches` good matches. A lower minimum also rejects true cuts between similar shots, and recall falls.

The minimum is chosen in code from the sweep below: the F1 is flat over a range of values, so we take the middle of the range of values that reach the best dev F1.

In [ ]:
import verification

# Frames of the dev videos (needed by the SIFT check); Part 1 freed them after computing the signals.
videos = {n: tp1.read_frames(f"{INPUT}/{n}.mp4") for n in DEV}

# Candidate cuts: the Part 1 histogram signal with its best threshold, merged like in tp1.py.
threshold = best["histogram"][0]
GAP = tp1.VERIFY_GAP
RATIO = tp1.LOWE_RATIO

match_list = {}
for name in DEV:
    signal = sigs[name]["histogram"]
    candidates = np.array(detect_cuts(signal, threshold))   # same merging of nearby detections as Part 1
    match_list[name] = (
        candidates,
        verification.compute_candidate_match_num(videos[name], candidates, gap=GAP, ratio=RATIO),
    )


def dev_scores(min_matches):
    """Precision, recall, F1 and flash false alarms on both dev videos. None = no SIFT check."""
    kept = {name: (candidates if min_matches is None else candidates[match_num < min_matches])
            for name, (candidates, match_num) in match_list.items()}
    counts = evaluate_cuts(kept)
    return (*metrics.precision_recall_f1(*counts["cut"]), counts["flash_fa"])


# The SIFT matches are computed once above, so sweeping every value is cheap.
sweep_scores = {m: dev_scores(m) for m in range(2, 81)}
best_f1 = max(scores[2] for scores in sweep_scores.values())
best_values = [m for m, scores in sweep_scores.items() if scores[2] >= best_f1 - 1e-9]
CHOSEN_MIN_MATCHES = int(np.median(best_values))             # middle of the range that reaches the best F1

print("min_matches  precision  recall     F1   flash-FA")
for m in range(5, 81, 5):
    precision, recall, f1, flashes = sweep_scores[m]
    print(f"{m:11d} {precision:10.3f} {recall:7.3f} {f1:7.3f} {flashes:2d}/{n_flashes}")
print(f"\nbest F1 {best_f1:.3f} for min_matches from {min(best_values)} to {max(best_values)}; "
      f"chosen: {CHOSEN_MIN_MATCHES}")

print("\nverification          precision  recall     F1   flash-FA")
for label, min_matches in (("without", None), (f"with ({CHOSEN_MIN_MATCHES} matches)", CHOSEN_MIN_MATCHES)):
    precision, recall, f1, flashes = dev_scores(min_matches)
    print(f"{label:22s} {precision:10.3f} {recall:7.3f} {f1:7.3f} {flashes:2d}/{n_flashes}")

#### Without verification means accepting every candidate without verifying if it passes the min matches test

In [ ]:
name = "dev_01"
candidates, match_num = match_list[name]
rejected = candidates[match_num >= CHOSEN_MIN_MATCHES]
gt_cuts = np.array([t["start"] for t in gts[name]["transitions"]])
false_alarms = [int(t) for t in rejected if np.all(np.abs(gt_cuts - t) > 2)]
t = false_alarms[0]
n_matches = int(match_num[np.where(candidates == t)[0][0]])

flash = next(
    (
        d for d in gts[name]["distractors"]
        if d["start"] - metrics.TOLERANCE <= t <= d["end"] + metrics.TOLERANCE
    ),
    None,
)
if flash is None:
    cause = (
        "The candidate is not a labelled flash. "
        "Frames on either side still match, so the histogram peak comes from "
        "motion or an object crossing the same shot."
    )
else:
    cause = (
        f"It falls on the flash distractor covering frames {flash['start']}\u2013{flash['end']}. "
        "The frames four steps away are outside that flash and still match."
    )

frames = videos[name]
good, kp_a, kp_b = verification.sift_match(frames[t - GAP], frames[t + GAP], RATIO)
gap_px = 24
shown = sorted(good, key=lambda m: m.distance)[:30]
left = cv2.copyMakeBorder(
    frames[t - GAP], 0, 0, 0, gap_px, cv2.BORDER_CONSTANT, value=(255, 255, 255)
)
vis = cv2.drawMatches(
    left, kp_a, frames[t + GAP], kp_b, shown, None,
    matchColor=(0, 180, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

print(
    f"Rejected candidate at frame {t} of {name}: {n_matches} good matches "
    f"between frames {t - GAP} and {t + GAP}."
)
print(cause)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(
    np.concatenate([
        frames[t - GAP],
        np.full((frames[t - GAP].shape[0], gap_px, 3), 255, np.uint8),
        frames[t + GAP],
    ], axis=1),
    cv2.COLOR_BGR2RGB,
))
axes[0].set_title(f"{name}: frames {t - GAP} and {t + GAP} around rejected candidate {t}")
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"{len(good)} matches after the ratio test; strongest {len(shown)} drawn")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Removed false alarm

The figure shows one candidate that the histogram detector proposes and the SIFT check rejects. The printed lines give its frame, how many matches survive the ratio test, and whether it lands on a labelled flash. The two frames are `t - 4` and `t + 4`. A large match count is the reason for the rejection: both frames still belong to the same shot.

## 4. Part 3 - Recurring scenes

Each shot is a ground-truth interval (`start` through `end`). One keyframe is the middle frame. Three keyframes are spread from `start` to `end`. Colour similarity is `1 − χ²`. SIFT similarity is the number of matches that pass Lowe's ratio test at 0.75. `max` keeps the best keyframe pair. `mean` averages every keyframe pair.

In [ ]:
import scenes

configs = [
    ("histogram", 1, "max"),
    ("histogram", 3, "max"),
    ("histogram", 3, "mean"),
    ("sift", 1, "max"),
    ("sift", 3, "max"),
    ("sift", 3, "mean"),
]
dev_matrices = {}
rows = []
for feature, n_keyframes, rule in configs:
    sims, shot_lists = [], []
    dev_matrices[(feature, n_keyframes, rule)] = {}
    for name in DEV:
        shots = gts[name]["shots"]
        sim = scenes.similarity_matrix(videos[name], shots, feature, n_keyframes, rule)
        dev_matrices[(feature, n_keyframes, rule)][name] = sim
        sims.append(sim)
        shot_lists.append(shots)
    threshold, precision, recall, f1 = scenes.best_threshold(sims, shot_lists)
    rows.append((feature, n_keyframes, rule, threshold, precision, recall, f1))

print("feature     keyframes  rule  threshold  precision  recall     F1")
for feature, n_keyframes, rule, threshold, precision, recall, f1 in rows:
    print(
        f"{feature:10s} {n_keyframes:9d} {rule:5s} {threshold:10.3f} "
        f"{precision:9.3f} {recall:7.3f} {f1:7.3f}"
    )

In [ ]:
chosen_rule = {}
for feature in ("histogram", "sift"):
    feature_rows = [row for row in rows if row[0] == feature and row[1] == 3]
    chosen_rule[feature] = max(feature_rows, key=lambda row: row[6])[2]

reported = []
for row in rows:
    feature, n_keyframes, rule = row[0], row[1], row[2]
    if n_keyframes == 1 and rule == "max":
        reported.append(row)
    elif n_keyframes == 3 and rule == chosen_rule[feature]:
        reported.append(row)

print("feature     keyframes  rule  threshold  precision  recall     F1")
for feature, n_keyframes, rule, threshold, precision, recall, f1 in reported:
    print(
        f"{feature:10s} {n_keyframes:9d} {rule:5s} {threshold:10.3f} "
        f"{precision:9.3f} {recall:7.3f} {f1:7.3f}"
    )

best_config = max(reported, key=lambda row: row[6])
feature, n_keyframes, rule, threshold = best_config[:4]
print(f"best: {feature}, {n_keyframes} keyframes, {rule}, threshold {threshold:.3f}")
print("rule by feature:", chosen_rule)

In [ ]:
feature, n_keyframes, rule, threshold, *_ = best_config
name = "dev_01"
sim = dev_matrices[(feature, n_keyframes, rule)][name]
gt_matrix = scenes.ground_truth_matrix(gts[name]["shots"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sim, cmap="magma")
axes[0].set_title(f"{feature}, {n_keyframes} keyframes, {rule}")
axes[1].imshow(gt_matrix, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground truth")
for ax in axes:
    ax.set_xlabel("Shot")
    ax.set_ylabel("Shot")
fig.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
import grouping

feature, n_keyframes, rule, threshold, *_ = best_config
name = "dev_01"
shots = gts[name]["shots"]
sim = dev_matrices[(feature, n_keyframes, rule)][name]
labels = grouping.group_shots(sim, threshold)
mids = [scenes.keyframe_indices(shot, 1)[0] for shot in shots]

cols = 8
n_rows = (len(shots) + cols - 1) // cols
fig, axes = plt.subplots(n_rows, cols, figsize=(16, 2.2 * n_rows))
axes = axes.ravel()
for ax, shot, label, mid in zip(axes, shots, labels, mids):
    ax.imshow(cv2.cvtColor(videos[name][mid], cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{shot['id']} / {label}",
        fontsize=8,
        color=plt.cm.tab20(label % 20),
    )
    ax.axis("off")
for ax in axes[len(shots):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

success = failure_miss = failure_merge = None
for a in range(len(shots)):
    for b in range(a + 1, len(shots)):
        same_gt = shots[a]["scene"] == shots[b]["scene"]
        same_pred = labels[a] == labels[b]
        if success is None and same_gt and same_pred:
            success = (a, b)
        if failure_miss is None and same_gt and not same_pred:
            failure_miss = (a, b)
        if failure_merge is None and (not same_gt) and same_pred:
            failure_merge = (a, b)

a, b = success
print(
    f"Success: shots {shots[a]['id']} and {shots[b]['id']} "
    f"are both {shots[a]['scene']} and both label {labels[a]}. "
    f"Middle frames {mids[a]} and {mids[b]}."
)
fail = failure_miss if failure_miss is not None else failure_merge
kind = "missed repeat" if failure_miss is not None else "false merge"
a, b = fail
print(
    f"Failure ({kind}): shots {shots[a]['id']} and {shots[b]['id']}, "
    f"scenes {shots[a]['scene']} and {shots[b]['scene']}, "
    f"labels {labels[a]} and {labels[b]}. "
    f"Middle frames {mids[a]} and {mids[b]}."
)

Shots 0 and 14 are both scene S00 and receive the same predicted label. The middle frames are 36 and 1574, so the grouping recovers that repeat.

Shots 27 and 28 belong to scenes S21 and S22 but share one predicted label. That failure is a false merge: the colour of the two shots is close enough that the best keyframe pair passes the threshold.

In [ ]:
import numbers


def same(a, b):
    return bool(np.isclose(a, b, rtol=1e-6)) if isinstance(a, numbers.Real) else a == b


def show(value):
    return f"{value:.10g}" if isinstance(value, numbers.Real) and not isinstance(value, bool) else str(value)


# (what it is, constant in tp1.py, value chosen in this notebook, value used by tp1.py)
checks = [
    ("histogram bins",      "HIST_BINS",           best_bins,             tp1.HIST_BINS),
    ("histogram threshold", "HISTOGRAM_THRESHOLD", best["histogram"][0],  tp1.HISTOGRAM_THRESHOLD),
    ("merge gap",           "MERGE_GAP",           CONFIG["merge_gap"],   tp1.MERGE_GAP),
    ("min SIFT matches",    "MIN_SIFT_MATCHES",    CHOSEN_MIN_MATCHES,    tp1.MIN_SIFT_MATCHES),
    ("scene feature",       "SCENE_FEATURE",       best_config[0],        tp1.SCENE_FEATURE),
    ("scene keyframes",     "SCENE_KEYFRAMES",     best_config[1],        tp1.SCENE_KEYFRAMES),
    ("scene rule",          "SCENE_RULE",          best_config[2],        tp1.SCENE_RULE),
    ("scene threshold",     "SCENE_THRESHOLD",     best_config[3],        tp1.SCENE_THRESHOLD),
]

all_ok = True
print(f"{'':5s}{'setting':22s} {'notebook':>14s}   tp1.py")
for label, constant, notebook_value, tp1_value in checks:
    ok = same(notebook_value, tp1_value)
    all_ok = all_ok and ok
    print(f"{'OK' if ok else 'DIFF':5s}{label:22s} {show(notebook_value):>14s}   {constant} = {show(tp1_value)}")

print()
print("tp1.py uses the same configuration as the notebook." if all_ok
      else "Copy the notebook values of the DIFF rows into tp1.py, then run tp1.py again.")

## 5. Results (development and test)

The table is computed below from the predictions that `python tp1.py` wrote to `../output`, using the same
`metrics` functions as the program. `tp1.py` uses the configuration chosen on the development videos (checked
against the notebook in the consistency cell above); the test videos are scored once, with that configuration.

In [ ]:
import glob

results = {}
for video_path in sorted(glob.glob(f"{INPUT}/*.mp4")):
    video = os.path.splitext(os.path.basename(video_path))[0]
    gt = json.load(open(f"{INPUT}/{video}.json"))
    prediction = json.load(open(f"{OUTPUT}/{video}.json"))
    results[video] = metrics.evaluate_video(gt, prediction)

dev = [results[v] for v in sorted(results) if v.startswith("dev")]
test = [results[v] for v in sorted(results) if v.startswith("test")]
metrics.print_table([(v, results[v]) for v in sorted(results)]
                    + [("DEV", metrics.sum_counts(dev)), ("TEST", metrics.sum_counts(test))])

## 6. Analysis and failure cases

### What Works

The grayscale histogram stays low during camera motion, which is why it beat pixel difference and the edge-change ratio in Part 1. SIFT verification then removes almost every flash: development flash false alarms fall to 1/12, and `dev_01` has none. For scenes, three keyframes with the best pair (max) is what holds the development scene F1 at 0.935. Recall stays high on the test videos (0.958), so repeated scenes are usually found.

### What Does not Work

From the final `tp1.py` output, we got three failures:

- Failure 1: Missed cut on frame 2947 on video `dev_01`. These are the similar-colour cuts: the histogram never reaches the threshold, so verification never sees them. Pixel difference and the edge-change ratio do react here, which is the limit of relying on colour.

In [ ]:
frames = videos["dev_01"]

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(cv2.cvtColor(
    np.concatenate([
        frames[2946],
        np.full((frames[2946].shape[0], gap_px, 3), 255, np.uint8),
        frames[2947],
    ], axis=1),
    cv2.COLOR_BGR2RGB,
))
ax.set_title("Frames 2946 & 2947 of video dev_01")
ax.axis("off")
plt.show()

- Failure 2: The flash on `dev_02` covering frames 1867 - 1869 is kept as a cut. The histogram crosses the threshold there. Verification compares frames 1863 and 1871, four frames away from the candidate at 1867, and those frames have only 3 matches after the ratio test. A candidate is rejected only when it has at least 15 matches, so this one stays. The flash changes the shot enough that the two frames no longer look like the same content.

In [ ]:
frames = videos["dev_02"]
gap_px = 24
t = 1867
left_i, right_i = t - GAP, t + GAP
good, kp_a, kp_b = verification.sift_match(frames[left_i], frames[right_i], RATIO)

print(
    f"{len(good)} matches between frames {left_i} and {right_i}; "
    f"a candidate is rejected only at {CHOSEN_MIN_MATCHES} or more."
)

shown = sorted(good, key=lambda m: m.distance)
left = cv2.copyMakeBorder(
    frames[left_i], 0, 0, 0, gap_px, cv2.BORDER_CONSTANT, value=(255, 255, 255)
)
vis = cv2.drawMatches(
    left, kp_a, frames[right_i], kp_b, shown, None,
    matchColor=(0, 180, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(
    np.concatenate([
        frames[left_i],
        np.full((frames[left_i].shape[0], gap_px, 3), 255, np.uint8),
        frames[right_i],
    ], axis=1),
    cv2.COLOR_BGR2RGB,
))
axes[0].set_title(f"dev_02: frames {left_i} and {right_i} around the flash at {t}")
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"{len(good)} matches after the ratio test")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

- Failure 3: Shots 27 and 28 of `dev_01` are scenes S21 and S22, but they receive the same label. With three keyframes and the max rule, the best pair is frame 2946 (last frame of shot 27) and frame 2988 (middle of shot 28). Their colour similarity is about 0.977, above the scene threshold of 0.950, so the two shots are linked. This is a false merge of different scenes that share almost the same colours. Every true repeat on `dev_01` was grouped, so the scene errors here are merges, not missed repeats.

In [ ]:
frames = videos["dev_01"]
gap_px = 24
left_i, right_i = 2946, 2988
similarity = scenes.color_similarity(
    signals.color_histogram(frames[left_i]),
    signals.color_histogram(frames[right_i]),
)
print(
    f"Shots 27 (S21) and 28 (S22): colour similarity {similarity:.3f} "
    f"between frames {left_i} and {right_i}, threshold {best_config[3]:.3f}."
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(cv2.cvtColor(
    np.concatenate([
        frames[left_i],
        np.full((frames[left_i].shape[0], gap_px, 3), 255, np.uint8),
        frames[right_i],
    ], axis=1),
    cv2.COLOR_BGR2RGB,
))
ax.set_title(f"dev_01: frame {left_i} (shot 27, S21) and frame {right_i} (shot 28, S22)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Conclusion and self-evaluation

## 8. Extras

### Extra - Adaptive threshold

Instead of one global threshold, a frame t is a cut when d(t) passes three tests:
- it is above a `floor`, which ignores noise in quiet shots;
- it is the maximum of its window of ±`half_window` frames;
- it is at least `min_ratio` times larger than the other values of the window. The frame right next to t is left
  out of this comparison, because a cut can stay high for two frames.

Detections at most 2 frames apart are merged into one cut, as in Part 1. We expect a flash to produce two peaks
of similar height (when it starts and when it ends), so a large `min_ratio` should reject it; the flash
false-alarm column shows whether this happens.

`half_window`, `min_ratio` and `floor` are chosen by grid search on the dev videos, separately for each signal,
and compared with the best global threshold of Part 1. The rule is also applied to the Part 1 combination: each
signal is divided by its Part 1 threshold and the minimum (AND) or maximum (OR) is taken, so a value above 1
means that the Part 1 rule fires. We test two variants: with the floor fixed at 1 (the Part 1 rule decides which
frames qualify and the peak test only removes some of them) and with the floor also searched.

In [ ]:
from itertools import product


def adaptive_cuts(d, *, half_window, min_ratio, floor, guard_frames=1, merge_gap=2):
    """Cuts of one signal: local maxima that are min_ratio times larger than the rest of their window."""
    mask = np.zeros(len(d), dtype=bool)
    for t in range(1, len(d)):
        lo, hi = max(0, t - half_window), t + half_window + 1
        if d[t] <= floor or d[t] < d[lo:hi].max():
            continue
        others = np.concatenate([d[lo:max(0, t - guard_frames)], d[t + guard_frames + 1:hi]])
        mask[t] = len(others) == 0 or d[t] >= min_ratio * others.max()
    return cuts_from_mask(mask=mask, score=d, merge_gap=merge_gap)


def adaptive_sweep(signal_by_video, *, half_windows=(3, 5, 8, 12), min_ratios=(1.5, 2, 3, 5), floors=None):
    """Best (half_window, min_ratio, floor) by dev F1."""
    if floors is None:
        floors = np.quantile(np.concatenate(list(signal_by_video.values())), np.linspace(0.85, 0.98, 5))
    results = []
    for half_window, min_ratio, floor in product(half_windows, min_ratios, floors):
        predictions = {video: adaptive_cuts(signal, half_window=half_window, min_ratio=min_ratio, floor=floor)
                       for video, signal in signal_by_video.items()}
        counts = evaluate_cuts(pred_by_video=predictions)
        p, r, f1 = metrics.precision_recall_f1(*counts["cut"])
        results.append(dict(half_window=half_window, min_ratio=min_ratio, floor=float(floor),
                            precision=p, recall=r, f1=f1, flash_fa=counts["flash_fa"]))
    return max(results, key=lambda res: res["f1"])


def combined_signal(signals_of_video, *, thresholds, mode):
    """Part 1 combination as one signal: above 1 means the Part 1 rule fires."""
    normalised = [signals_of_video[name] / T for name, T in thresholds.items()]
    return (np.min if mode == "and" else np.max)(normalised, axis=0)


def adaptive_row(signal, a, floor_text=None):
    floor_text = floor_text or f"{a['floor']:.3g}"
    parameters = f"m={a['half_window']} k={a['min_ratio']:.1f} floor={floor_text}"
    return (signal, "adaptive", parameters, a["precision"], a["recall"], a["f1"], a["flash_fa"])


def print_rows(rows, *, n_flashes):
    print(f"{'signal':10s} {'method':9s} {'parameters':34s} {'P':>6s} {'R':>6s} {'F1':>6s} {'flash-FA':>9s}")
    for signal, method, parameters, p, r, f1, flash_fa in rows:
        print(f"{signal:10s} {method:9s} {parameters:34s} {p:6.3f} {r:6.3f} {f1:6.3f} {flash_fa:>5d}/{n_flashes}")


# single signals: best global threshold vs best adaptive rule
rows = []
for s in SIGNALS:
    T, p, r, f1, fa = best[s]
    rows.append((s, "global", f"T={T:.4g}", p, r, f1, fa))
    rows.append(adaptive_row("", adaptive_sweep({v: sigs[v][s] for v in DEV})))
print_rows(rows, n_flashes=n_flashes)

combo_by_video = {v: combined_signal(sigs[v], thresholds=thresholds, mode=mode) for v in DEV}
base = evaluate_cuts(pred_by_video=predicted)
p, r, f1 = metrics.precision_recall_f1(*base["cut"])

print(f"\n{mode.upper()}: " + " + ".join(thresholds))
print_rows([
    ("combo", "global", "Part 1 thresholds", p, r, f1, base["flash_fa"]),
    adaptive_row("", adaptive_sweep(combo_by_video, floors=[1.0]), floor_text="1 (Part 1 gate)"),
    adaptive_row("", adaptive_sweep(combo_by_video)),
], n_flashes=n_flashes)

**Result:** the sliding-window rule beats the best global threshold for every signal on the dev videos (pixel F1 0.857 to 0.951, histogram 0.889 to 0.938, ECR 0.710 to 0.887) and, gated by the Part 1 combination, raises F1 from 0.914 to 0.965 and cuts flash false alarms from 12/12 to 4/12, but it still removes flashes less well than the SIFT verification of Part 2 (F1 0.971, 1/12).

### Extra - RANSAC inliers for the verification

In Part 2 a candidate is rejected when the frames t-4 and t+4 have at least `min_matches` good SIFT matches. Here
we count only the matches that are inliers of a homography estimated with RANSAC, so matches that do not agree
with a single geometric transformation no longer count. We compare both counts over a range of minimum values.

In [ ]:
def count_ransac_inliers(frame_a, frame_b, *, ratio, reproj_threshold=3.0):
    """Number of ratio-test matches that fit one homography (RANSAC inliers)."""
    good, kp_a, kp_b = verification.sift_match(frame_a, frame_b, ratio)
    if len(good) < 4:                                   # a homography needs at least 4 matches
        return 0
    src = np.float32([kp_a[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp_b[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    _, inliers = cv2.findHomography(src, dst, cv2.RANSAC, reproj_threshold)
    return 0 if inliers is None else int(inliers.sum())


def inliers_per_candidate(video, *, gap, ratio):
    """RANSAC inlier count between frames t-gap and t+gap, for every candidate t of the video."""
    frames = videos[video]
    last = len(frames) - 1
    return np.array([count_ransac_inliers(frames[max(t - gap, 0)], frames[min(t + gap, last)], ratio=ratio)
                     for t in match_list[video][0]])


def score_rejection(count_by_video, *, min_count):
    """P, R, F1 and flash false alarms when candidates with count >= min_count are rejected."""
    kept = {v: match_list[v][0][count_by_video[v] < min_count] for v in DEV}
    counts = evaluate_cuts(pred_by_video=kept)
    return (*metrics.precision_recall_f1(*counts["cut"]), counts["flash_fa"])


count_by_method = {
    "raw matches (Part 2)": {v: match_list[v][1] for v in DEV},
    "RANSAC inliers": {v: inliers_per_candidate(v, gap=GAP, ratio=RATIO) for v in DEV},
}

MIN_COUNTS = list(range(2, 81, 2))
scores = {method: [score_rejection(counts, min_count=m) for m in MIN_COUNTS]
          for method, counts in count_by_method.items()}

print(f"{'method':22s} {'best min':>8s} {'P':>6s} {'R':>6s} {'F1':>6s} {'flash-FA':>9s}")
for method, rows in scores.items():
    i = max(range(len(rows)), key=lambda j: rows[j][2])          # row with the best F1
    p, r, f1, flash_fa = rows[i]
    print(f"{method:22s} {MIN_COUNTS[i]:8d} {p:6.3f} {r:6.3f} {f1:6.3f} {flash_fa:>5d}/{n_flashes}")

fig, ax = plt.subplots(figsize=(6, 4))
for method, rows in scores.items():
    ax.plot(MIN_COUNTS, [row[2] for row in rows], marker=".", label=method)
ax.set_xlabel("minimum count to reject a candidate")
ax.set_ylabel("cut F1 (dev_01 + dev_02)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Result:** counting only RANSAC inliers reaches the same best dev F1 as raw matches (0.971, 1/12 flash false alarms) at a minimum of 8 instead of 16, so it gives no gain in score but works with a smaller minimum.

### Extra - Combining colour and local features for scene grouping

Shots are compared with the two similarities from Part 3, each computed with three keyframes per shot and the best
keyframe pair (`max`): a colour similarity (1 minus the chi-square distance between HSV histograms) and a
local-feature similarity (SIFT matches that pass the ratio test, divided by the largest count over both dev videos
so that it lies between 0 and 1 and one scale fits both videos). The shot similarity is a weighted sum, with
weight `w` on the colour part. For each `w` the grouping threshold is chosen on the dev videos, and the pairwise
scene F1 is compared with colour alone (`w = 1`) and SIFT alone (`w = 0`).

In [ ]:
KEYFRAMES, RULE = 3, "max"            
colour_sims = {v: dev_matrices[("histogram", KEYFRAMES, RULE)][v] for v in DEV}
sift_sims = {v: dev_matrices[("sift", KEYFRAMES, RULE)][v] for v in DEV}
sift_scale = max(sim.max() for sim in sift_sims.values())    
shot_lists = [gts[v]["shots"] for v in DEV]

COLOUR_WEIGHTS = [0, 0.25, 0.5, 0.75, 1]
rows = []
for w in COLOUR_WEIGHTS:
    sims = [w * colour_sims[v] + (1 - w) * sift_sims[v] / sift_scale for v in DEV]
    threshold, p, r, f1 = scenes.best_threshold(sims, shot_lists)
    rows.append((w, threshold, p, r, f1))

print(f"{'colour weight':>13s} {'threshold':>9s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for w, threshold, p, r, f1 in rows:
    print(f"{w:13.2f} {threshold:9.3f} {p:6.3f} {r:6.3f} {f1:6.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([row[0] for row in rows], [row[4] for row in rows], marker="o")
ax.set_xlabel("weight of the colour similarity (0 = SIFT only, 1 = colour only)")
ax.set_ylabel("pairwise scene F1 (dev_01 + dev_02)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Result:** a mix of colour and SIFT similarities reaches a pairwise scene F1 of 0.951 at colour weight 0.5, against 0.935 for colour alone and 0.931 for SIFT alone, a small gain of about 0.016 that may not carry over to the test videos because the weight and threshold are tuned on only two dev videos.

## 9. Use of AI tools